# 第15课：Attention 机制

## 学习目标
- 理解 Attention 机制解决的核心问题：序列建模中的信息瓶颈
- 掌握 Scaled Dot-Product Attention 的计算过程
- 理解 Query / Key / Value 的直觉含义
- 手动实现一个简单的 Attention 计算

## 核心概念：为什么需要 Attention？

上一课我们学了 RNN/LSTM，它们通过**逐步传递隐藏状态**来处理序列。

但有一个致命问题：**信息瓶颈**——不管输入序列多长，encoder 只能输出一个固定长度的向量，decoder 要从这个「压缩包」里还原所有信息。

**Attention 的直觉：**
与其把所有信息塞进一个向量，不如让模型在每一步都能「回头看」输入序列的所有位置，自动决定哪些位置更重要。

就像人翻译一句话时，不是先背下来再翻，而是边翻边回看原文的对应部分。

### Attention 在 AI 演进中的位置
- 2014-2015：Bahdanau Attention 用于机器翻译，解决 Seq2Seq 瓶颈
- 2017：Self-Attention → Transformer（下一课）
- 2018+：Attention 成为 BERT、GPT 等所有现代 LLM 的核心机制

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

## Scaled Dot-Product Attention

最核心的公式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Q / K / V 的直觉

| 角色 | 含义 | 图书馆类比 |
|------|------|------------|
| Query (Q) | 我在找什么 | 你的搜索词 |
| Key (K) | 每个位置的标签 | 每本书的标题/关键词 |
| Value (V) | 每个位置的实际内容 | 书的具体内容 |

计算过程：
1. Q 和 K 做点积 → 得到「相关性分数」
2. 除以 √d_k（缩放，防止梯度消失）
3. Softmax → 得到「注意力权重」（概率分布）
4. 权重乘以 V → 得到「加权信息」

In [ ]:
def softmax(x, axis=-1):
    """数值稳定的 softmax"""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    Scaled Dot-Product Attention
    Q: (seq_q, d_k)  K: (seq_k, d_k)  V: (seq_k, d_v)
    """
    d_k = Q.shape[-1]
    
    # 1. Q 和 K 的点积 → 相关性分数
    scores = Q @ K.T  # (seq_q, seq_k)
    
    # 2. 缩放
    scores = scores / np.sqrt(d_k)
    
    # 3. Softmax → 注意力权重
    weights = softmax(scores)  # (seq_q, seq_k)
    
    # 4. 加权求和
    output = weights @ V  # (seq_q, d_v)
    
    return output, weights

# 演示：4 个词的简单序列
np.random.seed(42)
d_k = 8  # 维度

# 模拟 4 个 token 的 Q, K, V
Q = np.random.randn(4, d_k) * 0.5
K = np.random.randn(4, d_k) * 0.5
V = np.random.randn(4, d_k) * 0.5

output, weights = scaled_dot_product_attention(Q, K, V)

print("注意力权重矩阵 (每行是 query 对所有 key 的注意力):")
print(np.round(weights, 3))
print(f"\n每行之和: {weights.sum(axis=1)}")  # 应该都是 1.0
print(f"\n输出形状: {output.shape}")

In [ ]:
# 可视化注意力权重
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(weights, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels([f'Key {i}' for i in range(4)])
ax.set_yticklabels([f'Query {i}' for i in range(4)])
ax.set_xlabel('Key 位置')
ax.set_ylabel('Query 位置')
ax.set_title('注意力权重热力图')

# 标注数值
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.colorbar(im, ax=ax, label='注意力权重')
plt.tight_layout()
plt.savefig('docs/attention_weights.png', dpi=100, bbox_inches='tight')
plt.show()
print("注意力热力图已保存")

In [ ]:
# 演示缩放的作用：d_k 很大时，不缩放会导致 softmax 趋近 one-hot
np.random.seed(42)

for d_k in [8, 64, 512]:
    Q = np.random.randn(1, d_k)
    K = np.random.randn(4, d_k)
    V = np.random.randn(4, d_k)
    
    # 不缩放
    scores_raw = (Q @ K.T)[0]
    weights_raw = softmax(scores_raw)
    
    # 缩放
    scores_scaled = scores_raw / np.sqrt(d_k)
    weights_scaled = softmax(scores_scaled)
    
    print(f"d_k={d_k:3d} | 未缩放 max权重: {weights_raw.max():.4f} | 缩放后 max权重: {weights_scaled.max():.4f}")

print("\n结论：不缩放时，维度越大 → softmax 越尖锐 → 梯度消失")

## 关键论文与项目

| 年份 | 论文/项目 | 意义 |
|------|-----------|------|
| 2014 | Bahdanau et al. "Neural Machine Translation by Jointly Learning to Align and Translate" | 首次提出 Attention 机制用于 NMT |
| 2015 | Luong Attention | 简化了 Attention 计算，提出 global/local attention |
| 2017 | Vaswani et al. "Attention Is All You Need" | Self-Attention → Transformer（下一课）|

## 总结

1. **Attention 解决的核心问题：** 序列到序列模型中的信息瓶颈
2. **核心操作：** Q·K^T → 缩放 → softmax → 加权 V
3. **缩放因子 √d_k：** 防止高维时 softmax 退化为 one-hot
4. **从 RNN+Attention 到纯 Attention：** Transformer 只差一步——Self-Attention

## 课后思考
1. 如果把 Attention 权重理解为「信息路由」，它和路由器转发有什么异同？
2. 为什么 Q/K/V 用不同的投影矩阵，而不是直接用原始输入？
3. Self-Attention 和 Cross-Attention 的区别是什么？各自适用什么场景？